In [3]:
!pip install faiss-cpu open_clip_torch accelerate qwen-vl-utils evaluate rouge_score bert_score scikit-learn datasets

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 78.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.5/35.5 MB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.2 MB/s eta 0:00:00
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=4df42141d945d3af928b6a5decc15aee2e77f16c5e9a9dcedce7b3ebc25fad70
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


In [4]:
import os
import random
import urllib.request
import zipfile
import torch
import torch.nn as nn
import torch.optim as optim
import open_clip
import faiss
import numpy as np
from datasets import load_dataset
from torch.utils.data import DataLoader
from PIL import Image, ImageFile
from google.colab import drive

# ==========================================
# 1. CẤU HÌNH & CHUẨN BỊ MÔI TRƯỜNG
# ==========================================
ImageFile.LOAD_TRUNCATED_IMAGES = True
drive.mount('/content/drive')

SAVE_DIR = "/content/drive/MyDrive/SLAKE_RAG-2"
IMG_DIR = "/content/SLAKE_imgs"
MODEL_NAME = 'hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224'
os.makedirs(SAVE_DIR, exist_ok=True)

def setup_slake_images():
    """Tải và giải nén dữ liệu ảnh SLAKE nếu chưa có"""
    if not os.path.exists(IMG_DIR):
        print("⬇️ Đang tải file ảnh gốc (imgs.zip) của SLAKE từ HuggingFace...")
        urllib.request.urlretrieve("https://huggingface.co/datasets/BoKelvin/SLAKE/resolve/main/imgs.zip", "imgs.zip")
        print("📦 Đang giải nén ảnh...")
        with zipfile.ZipFile("imgs.zip", 'r') as zipRef:
            zipRef.extractall(IMG_DIR)
        os.remove("imgs.zip")
        print("✅ Đã chuẩn bị xong kho ảnh SLAKE cục bộ!")

def get_image_path(img_name):
    """Lấy đường dẫn ảnh hợp lệ"""
    path1 = os.path.join(IMG_DIR, img_name)
    path2 = os.path.join(IMG_DIR, "imgs", img_name)
    if os.path.exists(path1): return path1
    if os.path.exists(path2): return path2
    return None

# ==========================================
# 2. ĐỊNH NGHĨA MÔ HÌNH (GATED FUSION)
# ==========================================
class GatedFusion(nn.Module):
    def __init__(self, embedDim=512):
        super(GatedFusion, self).__init__()
        self.gateLayer = nn.Linear(embedDim * 2, embedDim)
        self.sigmoidFunc = nn.Sigmoid()
        self.fusionLayer = nn.Linear(embedDim * 2, embedDim)

    def forward(self, imageEmbeds, textEmbeds):
        combinedEmbeds = torch.cat((imageEmbeds, textEmbeds), dim=1)
        gateValue = self.sigmoidFunc(self.gateLayer(combinedEmbeds))
        gatedImage = imageEmbeds * gateValue
        gatedText = textEmbeds * (1 - gateValue)
        fusedOutput = self.fusionLayer(torch.cat((gatedImage, gatedText), dim=1))
        return fusedOutput / fusedOutput.norm(dim=-1, keepdim=True)

# ==========================================
# 3. CLASS TRUY XUẤT (RETRIEVER)
# ==========================================
class SlakeFaissRetriever:
    """Class phục vụ tìm kiếm RAG sau khi đã Train và Build Database"""
    def __init__(self):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.clip_model, _, self.preprocess = open_clip.create_model_and_transforms(MODEL_NAME)
        self.tokenizer = open_clip.get_tokenizer(MODEL_NAME)
        self.clip_model.to(self.device).eval()

        self.fusion_model = GatedFusion(embedDim=512).to(self.device)
        self.fusion_model.load_state_dict(torch.load(f"{SAVE_DIR}/slake_gated_fusion.pth", map_location=self.device))
        self.fusion_model.eval()

        self.index = faiss.read_index(f"{SAVE_DIR}/slake_fusion.index")
        self.metadata = torch.load(f"{SAVE_DIR}/slake_metadata.pt")

    def retrieveTopK(self, raw_img, query_question, topK=4):
        proc_img = self.preprocess(raw_img).unsqueeze(0).to(self.device)
        tokens = self.tokenizer([query_question]).to(self.device)

        with torch.no_grad():
            img_feat = self.clip_model.encode_image(proc_img)
            txt_feat = self.clip_model.encode_text(tokens)
            img_feat /= img_feat.norm(dim=-1, keepdim=True)
            txt_feat /= txt_feat.norm(dim=-1, keepdim=True)
            fusion_vec = self.fusion_model(img_feat, txt_feat).cpu().numpy().astype('float32')

        distances, indices = self.index.search(fusion_vec, topK)
        results = []
        for idx in indices[0]:
            if idx != -1 and idx < len(self.metadata):
                results.append(self.metadata[idx])
        return results

# ==========================================
# 4. CÁC HÀM XỬ LÝ CHÍNH
# ==========================================
def train_slake_fusion(epochs=5):
    """Huấn luyện mô hình GatedFusion"""
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"\n🚀 [1] BẮT ĐẦU HUẤN LUYỆN TRÊN: {device}")

    bioMedClip, _, preprocessImage = open_clip.create_model_and_transforms(MODEL_NAME)
    tokenizer = open_clip.get_tokenizer(MODEL_NAME)
    bioMedClip.to(device).eval()

    fusionModel = GatedFusion(embedDim=512).to(device)
    optimizer = optim.AdamW(fusionModel.parameters(), lr=2e-4, weight_decay=1e-5)
    lossFunction = nn.CrossEntropyLoss()

    trainDS = load_dataset("BoKelvin/SLAKE", split="train")

    def collateBatch(batchData):
        imagesProc, questions, answers = [], [], []
        for item in batchData:
            if item.get('q_lang') != 'en': continue
            imgName = item.get('img_name')
            imgPath = get_image_path(imgName) if imgName else None
            if not imgPath: continue

            try:
                img = Image.open(imgPath).convert("RGB")
                imagesProc.append(preprocessImage(img))
                questions.append(str(item['question']))
                answers.append(str(item['answer']))
            except Exception:
                continue

        if not imagesProc: return None, None, None
        return torch.stack(imagesProc), tokenizer(questions), tokenizer(answers)

    dataLoader = DataLoader(trainDS, batch_size=64, shuffle=True, collate_fn=collateBatch, drop_last=True)

    for epoch in range(epochs):
        fusionModel.train()
        for batchIdx, (images, questions, answers) in enumerate(dataLoader):
            if images is None: continue
            images, questions, answers = images.to(device), questions.to(device), answers.to(device)
            optimizer.zero_grad()

            with torch.no_grad():
                imgFeats = bioMedClip.encode_image(images)
                questFeats = bioMedClip.encode_text(questions)
                ansFeats = bioMedClip.encode_text(answers)

                imgFeats /= imgFeats.norm(dim=-1, keepdim=True)
                questFeats /= questFeats.norm(dim=-1, keepdim=True)
                ansFeats /= ansFeats.norm(dim=-1, keepdim=True)

            fusionVec = fusionModel(imgFeats, questFeats)
            logits = (fusionVec @ ansFeats.T) / 0.07
            labels = torch.arange(len(images)).to(device)
            loss = (lossFunction(logits, labels) + lossFunction(logits.T, labels)) / 2

            loss.backward()
            optimizer.step()

            if batchIdx % 20 == 0:
                print(f"Epoch [{epoch+1}/{epochs}] - Batch {batchIdx}/{len(dataLoader)} - Loss: {loss.item():.4f}")

    torch.save(fusionModel.state_dict(), f"{SAVE_DIR}/slake_gated_fusion.pth")
    print("✅ Đã lưu thành công trọng số Gated Fusion!")

def build_faiss_database():
    """Xây dựng cơ sở dữ liệu Vector (FAISS)"""
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"\n🚀 [2] BẮT ĐẦU XÂY DỰNG VECTOR DATABASE TRÊN: {device}")

    bioMedClip, _, preprocessImage = open_clip.create_model_and_transforms(MODEL_NAME)
    tokenizer = open_clip.get_tokenizer(MODEL_NAME)
    bioMedClip.to(device).eval()

    fusionModel = GatedFusion(embedDim=512).to(device)
    fusionModel.load_state_dict(torch.load(f"{SAVE_DIR}/slake_gated_fusion.pth", map_location=device))
    fusionModel.eval()

    trainDS = load_dataset("BoKelvin/SLAKE", split="train")
    faissIndex = faiss.IndexFlatL2(512)
    metaDataList = []

    with torch.no_grad():
        for i, item in enumerate(trainDS):
            if item.get('q_lang') == 'zh': continue
            imgName = item.get('img_name')
            imgPath = get_image_path(imgName) if imgName else None
            if not imgPath: continue

            try:
                img = Image.open(imgPath).convert("RGB")
                procImg = preprocessImage(img).unsqueeze(0).to(device)
                tokens = tokenizer([str(item['question'])]).to(device)

                imgFeat = bioMedClip.encode_image(procImg)
                txtFeat = bioMedClip.encode_text(tokens)
                imgFeat /= imgFeat.norm(dim=-1, keepdim=True)
                txtFeat /= txtFeat.norm(dim=-1, keepdim=True)

                fusionVec = fusionModel(imgFeat, txtFeat).cpu().numpy().astype('float32')
                faissIndex.add(fusionVec)

                metaDataList.append({
                    "id": i,
                    "imgName": imgName,
                    "question": str(item['question']),
                    "answer": str(item['answer'])
                })
            except Exception:
                continue

            if len(metaDataList) % 500 == 0:
                print(f"Đã lập chỉ mục: {len(metaDataList)} mẫu tiếng Anh...")

    faiss.write_index(faissIndex, f"{SAVE_DIR}/slake_fusion.index")
    torch.save(metaDataList, f"{SAVE_DIR}/slake_metadata.pt")
    print("🎉 ĐÃ HOÀN THÀNH LƯU FAISS INDEX & METADATA!")

def check_actual_retrieval(num_samples=3):
    """Kiểm tra độ chính xác của hệ thống RAG"""
    print(f"\n🚀 [3] BẮT ĐẦU ĐỐI SOÁT TRUY XUẤT RAG (RETRIEVAL)")
    setup_slake_images()
    retriever = SlakeFaissRetriever()
    print("✅ Đã khởi tạo phiên bản SlakeFaissRetriever MỚI NHẤT!")

    rawTestDS = load_dataset("BoKelvin/SLAKE", split="test")
    englishTestDS = [item for item in rawTestDS if item.get('q_lang') != 'zh']
    sampleIndices = random.sample(range(len(englishTestDS)), min(num_samples, len(englishTestDS)))

    for i, idx in enumerate(sampleIndices):
        item = englishTestDS[idx]
        imgName = item.get('img_name')
        imgPath = get_image_path(imgName) if imgName else None

        print(f"\n================ MẪU THỬ KIỂM TRA #{i+1} (Test ID: {idx}) ================")
        if not imgPath:
            print(f"❌ Không tìm thấy tệp ảnh mục tiêu cục bộ: {imgName}")
            continue

        rawImg = Image.open(imgPath).convert("RGB")
        queryQuestion = str(item['question'])
        groundTruth = str(item['answer'])

        print(f"🎯 Câu hỏi mục tiêu: {queryQuestion}")
        print(f"✅ Đáp án gốc     : {groundTruth}")

        retrievedContexts = retriever.retrieveTopK(rawImg, queryQuestion, topK=4)
        print(f"📊 Số lượng ca bệnh FAISS trả về: {len(retrievedContexts)}")

        if not retrievedContexts:
            print("❌ CẢNH BÁO: Mạng kết nối FAISS trả về danh sách rỗng!")
        else:
            print("📚 Chi tiết các ca bệnh truy xuất được:")
            for rank, ctx in enumerate(retrievedContexts):
                ctxImg = ctx.get('imgName', 'Không xác định')
                print(f"  --> Top {rank+1} tương đồng:")
                print(f"      + Ảnh tham chiếu : {ctxImg}")
                print(f"      + Câu hỏi        : {ctx.get('question')}")
                print(f"      + Đáp án         : {ctx.get('answer')}")

# ==========================================
# 5. CHẠY TOÀN BỘ PIPELINE
# ==========================================
if __name__ == "__main__":
    setup_slake_images()

    # BƯỚC 1: Huấn luyện mô hình (Comment lại nếu đã train xong)
    train_slake_fusion(epochs=5)

    # BƯỚC 2: Tạo FAISS Database (Comment lại nếu đã tạo xong)
    build_faiss_database()

    # BƯỚC 3: Test thử truy xuất
    check_actual_retrieval(num_samples=3)

Mounted at /content/drive
⬇️ Đang tải file ảnh gốc (imgs.zip) của SLAKE từ HuggingFace...
📦 Đang giải nén ảnh...
✅ Đã chuẩn bị xong kho ảnh SLAKE cục bộ!

🚀 [1] BẮT ĐẦU HUẤN LUYỆN TRÊN: cuda


open_clip_config.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

open_clip_pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  784MB            

open_clip_pytorch_model.bin: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

... (5 outputs left out)

Generating train split:   0%|          | 0/9835 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2099 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2094 [00:00<?, ? examples/s]

Epoch [1/5] - Batch 0/153 - Loss: 3.5474
Epoch [1/5] - Batch 20/153 - Loss: 2.5170
Epoch [1/5] - Batch 40/153 - Loss: 1.7934
Epoch [1/5] - Batch 60/153 - Loss: 1.9647
Epoch [1/5] - Batch 80/153 - Loss: 1.4380
Epoch [1/5] - Batch 100/153 - Loss: 1.5402
Epoch [1/5] - Batch 120/153 - Loss: 1.8344
Epoch [1/5] - Batch 140/153 - Loss: 1.8730
Epoch [2/5] - Batch 0/153 - Loss: 1.2921
Epoch [2/5] - Batch 20/153 - Loss: 1.0541
Epoch [2/5] - Batch 40/153 - Loss: 1.6583
Epoch [2/5] - Batch 60/153 - Loss: 1.4083
Epoch [2/5] - Batch 80/153 - Loss: 1.5103
Epoch [2/5] - Batch 100/153 - Loss: 1.2079
Epoch [2/5] - Batch 120/153 - Loss: 1.1755
Epoch [2/5] - Batch 140/153 - Loss: 1.4910
Epoch [3/5] - Batch 0/153 - Loss: 1.3211
Epoch [3/5] - Batch 20/153 - Loss: 1.1657
Epoch [3/5] - Batch 40/153 - Loss: 1.2500
Epoch [3/5] - Batch 60/153 - Loss: 1.4743
Epoch [3/5] - Batch 80/153 - Loss: 1.3309
Epoch [3/5] - Batch 100/153 - Loss: 1.4863
Epoch [3/5] - Batch 120/153 - Loss: 1.2985
Epoch [3/5] - Batch 140/153 -